# PO EXTRACTION in Flipkart

In [1]:
import os
import pandas as pd
import re
import hashlib

# ==========================================================
# 1. INPUT & OUTPUT PATHS
# ==========================================================
INPUT_DIR = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\flipkart\PO"
OUTPUT_DIR = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\Visual Studio Codes\Shubham Projects\Quick_Comm_Project"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================================
# 2. UNIVERSAL COLUMN NAME MAPPING
# ==========================================================
column_mapping = {
    "S. no.": ["s. no", "sl no", "sr no", "serial"],
    "HSN/SA Code": ["hsn", "hsn/sa code", "hsn code", "hsn/sac"],
    "FSN/ISBN13": ["fsn", "fsn/isbn13", "isbn", "fsn code"],
    "Quantity": ["quantity", "qty", "order qty"],
    "Pending Quantity": ["pending qty", "pending quantity"],
    "UOM": ["uom", "unit"],
    "Title": ["item description", "title", "product title"],
    "Required by Date": ["required by date"],
    "Supplier MRP": ["mrp", "supplier mrp"],
    "Supplier Price": ["supplier price", "price", "rate"],
    "Taxable Value": ["taxable value"],
    "IGST Rate": ["igst rate", "igst%", "igst"],
    "IGST Amount(per unit)": ["igst amount(per unit)", "igst amount"],
    "SGST Rate": ["sgst/utgst rate", "sgst rate"],
    "SGST Amount(per unit)": ["sgst amount(per unit)"],
    "CGST Rate": ["cgst rate"],
    "CGST Amount(per unit)": ["cgst amount(per unit)"],
    "CESS Rate": ["cess rate"],
    "CESS Amount(per unit)": ["cess amount(per unit)"],
    "Tax Amount": ["tax amount"],
    "Total Amount": ["total amount", "total"]
}

# ==========================================================
# 3. FILE HASH (DUPLICATE FILE CHECK)
# ==========================================================
def file_md5(path):
    hasher = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

# ==========================================================
# 4. HEADER ROW DETECTION
# ==========================================================
def detect_header_row(df):
    for i in range(0, 60):
        row_text = " ".join(str(x).lower() for x in df.iloc[i] if pd.notna(x))
        if "fsn" in row_text and "hsn" in row_text and "quantity" in row_text:
            return i
    return None

# ==========================================================
# 5. METADATA EXTRACTION
# ==========================================================
def extract_metadata(df):
    text = " ".join(df.astype(str).fillna("").values.flatten()).lower()

    po_number = re.search(r'po[#:\s]+([a-z0-9]+)', text)
    po_date = re.search(r'order\s*date[:\s]*([0-9/.\-]+)', text)
    po_expiry = re.search(r'(expiry\s*date|po\s*expiry)[:\s]*([0-9/.\-]+)', text)

    return (
        po_number.group(1) if po_number else "",
        po_date.group(1) if po_date else "",
        po_expiry.group(2) if po_expiry else ""
    )
def extract_po_addresses(df):
    text_lines = (
        df.astype(str)
        .fillna("")
        .values.flatten()
    )

    text = " ".join(text_lines)
    text = re.sub(r"\s+", " ", text)

    delivered_by = ""
    delivered_to = ""

    # -------- BILLED BY → PO Delivered By --------
    by_match = re.search(
        r"billed\s+by\s+(.*?)(gstin|state\s+code|shipped\s+from|billed\s+to\s+address)",
        text,
        re.IGNORECASE
    )
    if by_match:
        delivered_by = by_match.group(1).strip()

    # -------- BILLED TO ADDRESS → PO Delivered To --------
    to_match = re.search(
        r"billed\s+to\s+address\s+(.*?)(gstin|state\s+code|payment\s+details|mode\s+of\s+payment)",
        text,
        re.IGNORECASE
    )
    if to_match:
        delivered_to = to_match.group(1).strip()

    return delivered_by, delivered_to

# ==========================================================
# 6. PROCESS SINGLE PO FILE
# ==========================================================
def process_po_file(path):
    raw_df = pd.read_excel(path, header=None)
    po_num, po_date, po_expiry = extract_metadata(raw_df)
    po_delivered_by, po_delivered_to = extract_po_addresses(raw_df)
    header_row = detect_header_row(raw_df)
    if header_row is None:
        return [], po_num

    df = pd.read_excel(path, header=header_row).fillna("")
    df.columns = df.columns.astype(str).str.lower().str.strip()

    records = df.to_dict(orient="records")
    for r in records:
        r["po number"] = po_num
        r["po date"] = po_date
        r["po expiry"] = po_expiry
        r["po delivered by address"] = po_delivered_by
        r["po delivered to"] = po_delivered_to
    return records, po_num

# ==========================================================
# 7. LOAD FILES FROM INPUT DIRECTORY
# ==========================================================
excel_paths = [
    os.path.join(INPUT_DIR, f)
    for f in os.listdir(INPUT_DIR)
    if f.lower().endswith((".xlsx", ".xls"))
]

print(f"📂 Total Excel files found in input folder: {len(excel_paths)}")

if not excel_paths:
    print("❌ No Excel files found.")
    exit()


all_data = []
processed_hashes = set()
processed_po_numbers = set()

# ==========================================================
# 8. MAIN LOOP
# ==========================================================
for path in excel_paths:
    file_hash = file_md5(path)
    if file_hash in processed_hashes:
        print(f"🔁 Duplicate FILE skipped: {os.path.basename(path)}")
        continue

    processed_hashes.add(file_hash)

    preview_df = pd.read_excel(path, header=None)
    po_num, _, _ = extract_metadata(preview_df)

    if po_num in processed_po_numbers:
        print(f"🔁 Duplicate PO skipped: {po_num}")
        continue

    processed_po_numbers.add(po_num)

    print(f"📄 Processing: {os.path.basename(path)} | PO: {po_num}")
    rows, _ = process_po_file(path)
    all_data.extend(rows)

# ==========================================================
# 9. BUILD FINAL DATAFRAME
# ==========================================================
df_all = pd.DataFrame(all_data)
df_all.columns = df_all.columns.str.lower()

final_df = pd.DataFrame()

for final_col, patterns in column_mapping.items():
    matched = next((c for c in df_all.columns if any(p in c for p in patterns)), None)
    final_df[final_col] = df_all[matched] if matched else ""

final_df["PO Number"] = df_all.get("po number", "")
final_df["PO Date"] = df_all.get("po date", "")
final_df["PO Expiry"] = df_all.get("po expiry", "")
final_df["PO Delivered By Address"] = df_all.get("po delivered by address", "")
final_df["PO Delivered To"] = df_all.get("po delivered to", "")

# ==========================================================
# 10. FILTER VALID PRODUCT ROWS
# ==========================================================
def is_valid(row):
    return str(row["Quantity"]).replace(".", "", 1).isdigit()

final_df = final_df[final_df.apply(is_valid, axis=1)].reset_index(drop=True)

# ==========================================================
# 11. SAVE & DISPLAY
# ==========================================================
# output_file = os.path.join(OUTPUT_DIR, "Flipkart_All_POs.xlsx")
# final_df.to_excel(output_file, index=False)

pd.set_option("display.max_columns", None)
print("\n📊 Final DataFrame Preview:")
print(final_df.head(10))

print("\n🎉 COMPLETED SUCCESSFULLY")
# print("📁 Output saved at:", output_file)
print("📊 Total rows:", len(final_df))



📂 Total Excel files found in input folder: 445
📄 Processing: purchase_order_FAGWN07808591.xls | PO: fagwn07808591
📄 Processing: purchase_order_FAGWN07934808.xls | PO: fagwn07934808
📄 Processing: purchase_order_FASWN07894271.xls | PO: faswn07894271
📄 Processing: purchase_order_FASWN07894301.xls | PO: faswn07894301
📄 Processing: purchase_order_FASWN07900186.xls | PO: faswn07900186
📄 Processing: purchase_order_FASWN07900199.xls | PO: faswn07900199
📄 Processing: purchase_order_FBHWN05164815.xls | PO: fbhwn05164815
📄 Processing: purchase_order_FBHWN05164839.xls | PO: fbhwn05164839
📄 Processing: purchase_order_FBHWN05417273.xls | PO: fbhwn05417273
📄 Processing: purchase_order_FBHWN05417291.xls | PO: fbhwn05417291
📄 Processing: purchase_order_FBHWN05427883.xls | PO: fbhwn05427883
📄 Processing: purchase_order_FBHWN05428553.xls | PO: fbhwn05428553
📄 Processing: purchase_order_FBHWN05434758.xls | PO: fbhwn05434758
📄 Processing: purchase_order_FBHWN05458082.xls | PO: fbhwn05458082
📄 Processing: p

In [2]:
final_df

,S. no.,HSN/SA Code,FSN/ISBN13,Quantity,Pending Quantity,UOM,Title,Required by Date,Supplier MRP,Supplier Price,Taxable Value,IGST Rate,IGST Amount(per unit),SGST Rate,SGST Amount(per unit),CGST Rate,CGST Amount(per unit),CESS Rate,CESS Amount(per unit),Tax Amount,Total Amount,PO Number,PO Date,PO Expiry,PO Delivered By Address,PO Delivered To
0,1,48237010.0,BWLH5FZYW9XZEX9F,192,192.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-10,119.0 INR,51.35 INR,9390.720000000001 INR,0.0%,0.0 INR,2.5%,,2.5%,1.22 INR,0.0%,0.0 INR,468.48 INR,9859.20,fagwn07808591,11-03-26,10-04-26,"nan 1st Phase, Plot No. 162-A, Road Number 33 ...","nan Flipkart India Private Limited, Survey No ..."
1,1,48237010.0,BWLH5FZYW9XZEX9F,128,128.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-23,119.0 INR,48.3 INR,5888.0 INR,0.0%,0.0 INR,2.5%,,2.5%,1.15 INR,0.0%,0.0 INR,294.4 INR,6182.40,fagwn07934808,01-04-26,23-04-26,"nan 1st Phase, Plot No. 162-A, Road Number 33 ...","nan Flipkart India Private Limited, Survey No ..."
2,1,48237010,BWLH5FZYW9XZEX9F,12,12.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-12,119.0 INR,48.3 INR,551.9999999999999 INR,5.0%,2.3 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,27.599999999999998 INR,579.60,faswn07894271,26-03-26,13-04-26,"nan Ground Floor, 1, Pinnac Apartment Bamboo G...","nan Flipkart India Private Limited, Hillway In..."
3,2,48237010,BWLH4H8YSGDZZX78,24,24.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-04-12,89.0 INR,35.0 INR,799.92 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,40.08 INR,840.00,faswn07894271,26-03-26,13-04-26,"nan Ground Floor, 1, Pinnac Apartment Bamboo G...","nan Flipkart India Private Limited, Hillway In..."
4,3,46021919aa,PTDGXQBHGS5QPZZQ,12,12.0,pcs,ECO SOUL Square Palm Leaf 10 inch 10 Dinner Plate,2026-04-12,199.0 INR,118.3 INR,1419.6 INR,0.0%,0.0 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,0.0 INR,1419.60,faswn07894271,26-03-26,13-04-26,"nan Ground Floor, 1, Pinnac Apartment Bamboo G...","nan Flipkart India Private Limited, Hillway In..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,2,48236900.0,GLSH6KK67APTENYF,200,200.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-22,129.0 INR,62.3 INR,10560.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,1900.0 INR,12460.00,fuswn08080464,29-04-26,22-05-26,"nan Billing Address: 305, Udyog Kendra, Extens...","nan Flipkart India Private Limited, ESR Wareho..."
1151,1,48236900.0,GLSH6KK67APTENYF,320,320.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-24,129.0 INR,62.3 INR,16896.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,3040.0 INR,19936.00,fuswn08095377,01-05-26,25-05-26,"nan Billing Address: 305, Udyog Kendra, Extens...","nan Flipkart India Private Limited, ESR Wareho..."
1152,1,48237010.0,BWLH4H8YSGDZZX78,80,80.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-05-22,89.0 INR,35.0 INR,2666.4 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,133.6 INR,2800.00,fuswn08122617,07-05-26,22-05-26,"nan Billing Address: 305, Udyog Kendra, Extens...","nan Flipkart India Private Limited, ESR Wareho..."
1153,1,44191900.0,SPNH46S4G7HG3FWA,200,200.0,pcs,ECO SOUL_140 mm Biodegradable_Soup Spoon_Beige...,2026-07-04,99.0 INR,45.0 INR,8572.0 INR,5.0%,2.14 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,428.0 INR,9000.00,fuswn08276251,10-06-26,06-07-26,"nan Billing Address: 305, Udyog Kendra, Extens...","nan Flipkart India Private Limited, ESR Wareho..."


In [3]:
final_df['PO Date'] = pd.to_datetime(
    final_df['PO Date'],
    format='%d-%m-%y',
    errors='coerce'
)
final_df = final_df[final_df['PO Date'].dt.year != 2024]



# filter out 2024 year

In [4]:
final_df

,S. no.,HSN/SA Code,FSN/ISBN13,Quantity,Pending Quantity,UOM,Title,Required by Date,Supplier MRP,Supplier Price,Taxable Value,IGST Rate,IGST Amount(per unit),SGST Rate,SGST Amount(per unit),CGST Rate,CGST Amount(per unit),CESS Rate,CESS Amount(per unit),Tax Amount,Total Amount,PO Number,PO Date,PO Expiry,PO Delivered By Address,PO Delivered To
0,1,48237010.0,BWLH5FZYW9XZEX9F,192,192.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-10,119.0 INR,51.35 INR,9390.720000000001 INR,0.0%,0.0 INR,2.5%,,2.5%,1.22 INR,0.0%,0.0 INR,468.48 INR,9859.20,fagwn07808591,2026-03-11,10-04-26,"nan 1st Phase, Plot No. 162-A, Road Number 33 ...","nan Flipkart India Private Limited, Survey No ..."
1,1,48237010.0,BWLH5FZYW9XZEX9F,128,128.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-23,119.0 INR,48.3 INR,5888.0 INR,0.0%,0.0 INR,2.5%,,2.5%,1.15 INR,0.0%,0.0 INR,294.4 INR,6182.40,fagwn07934808,2026-04-01,23-04-26,"nan 1st Phase, Plot No. 162-A, Road Number 33 ...","nan Flipkart India Private Limited, Survey No ..."
2,1,48237010,BWLH5FZYW9XZEX9F,12,12.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-12,119.0 INR,48.3 INR,551.9999999999999 INR,5.0%,2.3 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,27.599999999999998 INR,579.60,faswn07894271,2026-03-26,13-04-26,"nan Ground Floor, 1, Pinnac Apartment Bamboo G...","nan Flipkart India Private Limited, Hillway In..."
3,2,48237010,BWLH4H8YSGDZZX78,24,24.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-04-12,89.0 INR,35.0 INR,799.92 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,40.08 INR,840.00,faswn07894271,2026-03-26,13-04-26,"nan Ground Floor, 1, Pinnac Apartment Bamboo G...","nan Flipkart India Private Limited, Hillway In..."
4,3,46021919aa,PTDGXQBHGS5QPZZQ,12,12.0,pcs,ECO SOUL Square Palm Leaf 10 inch 10 Dinner Plate,2026-04-12,199.0 INR,118.3 INR,1419.6 INR,0.0%,0.0 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,0.0 INR,1419.60,faswn07894271,2026-03-26,13-04-26,"nan Ground Floor, 1, Pinnac Apartment Bamboo G...","nan Flipkart India Private Limited, Hillway In..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,2,48236900.0,GLSH6KK67APTENYF,200,200.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-22,129.0 INR,62.3 INR,10560.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,1900.0 INR,12460.00,fuswn08080464,2026-04-29,22-05-26,"nan Billing Address: 305, Udyog Kendra, Extens...","nan Flipkart India Private Limited, ESR Wareho..."
1151,1,48236900.0,GLSH6KK67APTENYF,320,320.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-24,129.0 INR,62.3 INR,16896.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,3040.0 INR,19936.00,fuswn08095377,2026-05-01,25-05-26,"nan Billing Address: 305, Udyog Kendra, Extens...","nan Flipkart India Private Limited, ESR Wareho..."
1152,1,48237010.0,BWLH4H8YSGDZZX78,80,80.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-05-22,89.0 INR,35.0 INR,2666.4 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,133.6 INR,2800.00,fuswn08122617,2026-05-07,22-05-26,"nan Billing Address: 305, Udyog Kendra, Extens...","nan Flipkart India Private Limited, ESR Wareho..."
1153,1,44191900.0,SPNH46S4G7HG3FWA,200,200.0,pcs,ECO SOUL_140 mm Biodegradable_Soup Spoon_Beige...,2026-07-04,99.0 INR,45.0 INR,8572.0 INR,5.0%,2.14 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,428.0 INR,9000.00,fuswn08276251,2026-06-10,06-07-26,"nan Billing Address: 305, Udyog Kendra, Extens...","nan Flipkart India Private Limited, ESR Wareho..."


In [5]:
def clean_nan_words(x):
    if isinstance(x, str):
        # remove standalone 'nan' (case-insensitive)
        x = re.sub(r"\bnan\b", "", x, flags=re.IGNORECASE)
        # normalize spaces
        x = re.sub(r"\s+", " ", x).strip()
    return x

final_df["PO Delivered By Address"] = final_df["PO Delivered By Address"].apply(clean_nan_words)
final_df["PO Delivered To"] = final_df["PO Delivered To"].apply(clean_nan_words)


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\1926778745.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df["PO Delivered By Address"] = final_df["PO Delivered By Address"].apply(clean_nan_words)
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\1926778745.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df["PO Delivered To"] = final_df["PO Delivered To"].apply(clean_nan_words)


In [6]:
final_df

,S. no.,HSN/SA Code,FSN/ISBN13,Quantity,Pending Quantity,UOM,Title,Required by Date,Supplier MRP,Supplier Price,Taxable Value,IGST Rate,IGST Amount(per unit),SGST Rate,SGST Amount(per unit),CGST Rate,CGST Amount(per unit),CESS Rate,CESS Amount(per unit),Tax Amount,Total Amount,PO Number,PO Date,PO Expiry,PO Delivered By Address,PO Delivered To
0,1,48237010.0,BWLH5FZYW9XZEX9F,192,192.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-10,119.0 INR,51.35 INR,9390.720000000001 INR,0.0%,0.0 INR,2.5%,,2.5%,1.22 INR,0.0%,0.0 INR,468.48 INR,9859.20,fagwn07808591,2026-03-11,10-04-26,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Survey No 518,..."
1,1,48237010.0,BWLH5FZYW9XZEX9F,128,128.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-23,119.0 INR,48.3 INR,5888.0 INR,0.0%,0.0 INR,2.5%,,2.5%,1.15 INR,0.0%,0.0 INR,294.4 INR,6182.40,fagwn07934808,2026-04-01,23-04-26,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Survey No 518,..."
2,1,48237010,BWLH5FZYW9XZEX9F,12,12.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-12,119.0 INR,48.3 INR,551.9999999999999 INR,5.0%,2.3 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,27.599999999999998 INR,579.60,faswn07894271,2026-03-26,13-04-26,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust..."
3,2,48237010,BWLH4H8YSGDZZX78,24,24.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-04-12,89.0 INR,35.0 INR,799.92 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,40.08 INR,840.00,faswn07894271,2026-03-26,13-04-26,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust..."
4,3,46021919aa,PTDGXQBHGS5QPZZQ,12,12.0,pcs,ECO SOUL Square Palm Leaf 10 inch 10 Dinner Plate,2026-04-12,199.0 INR,118.3 INR,1419.6 INR,0.0%,0.0 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,0.0 INR,1419.60,faswn07894271,2026-03-26,13-04-26,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,2,48236900.0,GLSH6KK67APTENYF,200,200.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-22,129.0 INR,62.3 INR,10560.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,1900.0 INR,12460.00,fuswn08080464,2026-04-29,22-05-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin..."
1151,1,48236900.0,GLSH6KK67APTENYF,320,320.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-24,129.0 INR,62.3 INR,16896.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,3040.0 INR,19936.00,fuswn08095377,2026-05-01,25-05-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin..."
1152,1,48237010.0,BWLH4H8YSGDZZX78,80,80.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-05-22,89.0 INR,35.0 INR,2666.4 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,133.6 INR,2800.00,fuswn08122617,2026-05-07,22-05-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin..."
1153,1,44191900.0,SPNH46S4G7HG3FWA,200,200.0,pcs,ECO SOUL_140 mm Biodegradable_Soup Spoon_Beige...,2026-07-04,99.0 INR,45.0 INR,8572.0 INR,5.0%,2.14 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,428.0 INR,9000.00,fuswn08276251,2026-06-10,06-07-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin..."


# Invoice Extraction in Flipkart

In [7]:
import pdfplumber
import re
import os
import hashlib
import pandas as pd

# =======================================================
# 1. INPUT & OUTPUT PATHS
# =======================================================
INPUT_DIR = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\flipkart\Invoice"
OUTPUT_DIR = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Desktop\flipkart_code"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =======================================================
# 2. HELPER FUNCTIONS
# =======================================================
def clean_text(t):
    t = re.sub(r"\s+", " ", t or "").strip()
    t = re.sub(r"\b(nan|null|none)\b", "", t, flags=re.I)
    return t.strip()

def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

# =======================================================
# 3. PO NUMBER EXTRACTION
# =======================================================
def extract_po_number(text, filename=""):
    patterns = [
        r"P\.?\s*O\.?\s*#?\s*[:\-]\s*([A-Z0-9]{8,})",
        r"PO\s*No\.?\s*[:\-]\s*([A-Z0-9]{8,})",
        r"(FUSWN[A-Z0-9]+)"
    ]
    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return m.group(1)
    return ""

# =======================================================
# 4. ADDRESS EXTRACTION (ROBUST)
# =======================================================
def extract_invoice_metadata(text, filename=""):
    text = clean_text(text)

    inv = re.search(r"Invoice\s*(?:Number|No\.?)\s*[:\-]?\s*([A-Z0-9\/\-]+)", text, re.I)
    date = re.search(r"Invoice\s*Date\s*[:\-]?\s*([\d\/\-]+)", text, re.I)

    invoice_no = inv.group(1) if inv else ""
    invoice_date = date.group(1) if date else ""

    po_number = extract_po_number(text, filename)

    bill_to = ""
    ship_to = ""
    dispatch_from = ""

    # Pattern 1: Separate Bill To / Ship To blocks
    bill_match = re.search(r"Bill To\s*(.+?)(?=Ship To|Dispatch From|(?=GSTIN\b.+?Warehouse)|table|\n{3,})", text, re.S | re.I)
    if bill_match:
        bill_to = clean_text(bill_match.group(1))

    ship_match = re.search(r"Ship To\s*(.+?)(?=Dispatch From|GSTIN\b.+?Warehouse|table|\n{3,}|\Z)", text, re.S | re.I)
    if ship_match:
        ship_to = clean_text(ship_match.group(1))

    # Pattern 2: Dispatch From
    dispatch_match = re.search(r"Dispatch From\s*(.+?)(?=(Bill To|GSTIN\b|Invoice Number|table|\n{3,}|\Z))", text, re.S | re.I)
    if dispatch_match:
        dispatch_from = clean_text(dispatch_match.group(1))

    # Pattern 3: Combined "Bill To Ship To" (matches your sample)
    combined_match = re.search(r"Bill To\s+Ship To\s*(.+?)(?=(?=GSTIN\b.+?Warehouse)|table|Item Description|\n{3,}|\Z)", text, re.S | re.I)
    if combined_match:
        full_addr = clean_text(combined_match.group(1))
        if "GSTIN" in full_addr:
            # Split if GSTIN separates bill/ship (rare), else assign to both
            parts = re.split(r'(GSTIN \w+)', full_addr, 1)
            bill_to = clean_text(parts[0])
            ship_to = clean_text(full_addr)  # Full for ship_to as fallback
        else:
            bill_to = ship_to = full_addr

    # Pattern 4: Standalone "Warehouse" addresses (Flipkart common)
    if not ship_to and "Warehouse" in text:
        warehouse_match = re.search(r"(?:Warehouse|Flipkart)\s*,?\s*(.+?)(?=(GSTIN|HSN|Qty|table|\n{3,}|\Z))", text, re.S | re.I)
        if warehouse_match:
            ship_to = bill_to = clean_text(warehouse_match.group(1))

    # Clean up addresses: remove trailing GSTIN if duplicated
    for addr in [bill_to, ship_to, dispatch_from]:
        if addr:
            addr = re.sub(r'GSTIN\s*\w+\s*$', '', addr, flags=re.I).strip()

    return invoice_no, invoice_date, po_number, bill_to, ship_to, dispatch_from

# =======================================================
# 5. METADATA EXTRACTION
# =======================================================

# =======================================================
# 6. ITEM TABLE EXTRACTION (GST SAFE)
# =======================================================
def extract_invoice_items(pdf):
    items = []

    for page in pdf.pages:
        tables = page.extract_tables() or []
        for table in tables:
            table = [[clean_text(c) for c in row] for row in table if row]
            header = next((r for r in table if any("qty" in c.lower() for c in r)), None)
            if not header:
                continue

            headers = [h.lower() for h in header]
            start = table.index(header) + 1

            def idx(key):
                return next((i for i, h in enumerate(headers) if key in h), -1)

            has_igst = any("igst" in h for h in headers)
            has_cgst = any("cgst" in h for h in headers)
            has_sgst = any("sgst" in h for h in headers)

            for row in table[start:]:
                if not any(row):
                    continue

                item = {
                    "Invoice S.No": row[0] if row[0].isdigit() else "",
                    "Invoice Item & Description": row[idx("description")] if idx("description") >= 0 else "",
                    "Invoice SKU No": row[idx("sku")] if idx("sku") >= 0 else "",
                    "Invoice HSN/SAC": row[idx("hsn")] if idx("hsn") >= 0 else "",
                    "Invoice Case Qty": row[idx("qty")] if idx("qty") >= 0 else "",
                    "Invoice Rate": row[idx("rate")] if idx("rate") >= 0 else "",
                    "Invoice Amount": row[idx("amount")] if idx("amount") >= 0 else "",
                    "IGST %": "", "IGST Amount": "",
                    "CGST %": "", "CGST Amount": "",
                    "SGST %": "", "SGST Amount": ""
                }

                if has_igst:
                    i = idx("igst")
                    if i >= 0 and i + 1 < len(row):
                        item["IGST %"] = row[i]
                        item["IGST Amount"] = row[i + 1]

                if has_cgst and has_sgst:
                    c = idx("cgst")
                    s = idx("sgst")
                    if c >= 0:
                        item["CGST %"] = row[c]
                        item["CGST Amount"] = row[c + 1]
                    if s >= 0:
                        item["SGST %"] = row[s]
                        item["SGST Amount"] = row[s + 1]

                if item["Invoice Item & Description"]:
                    items.append(item)

    return items

# =======================================================
# 7. MAIN PROCESS
# =======================================================
all_data = []
processed = set()

pdf_files = [os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR) if f.lower().endswith(".pdf")]
print("📂 Total PDFs:", len(pdf_files))

for file in pdf_files:
    if file_md5(file) in processed:
        continue
    processed.add(file_md5(file))

    print("📄 Processing:", os.path.basename(file))

    with pdfplumber.open(file) as pdf:
        text = "\n".join(p.extract_text() or "" for p in pdf.pages)

        inv_no, inv_date, po_no, bill_to, ship_to, dispatch_from = extract_invoice_metadata(
            text, os.path.basename(file)
        )

        items = extract_invoice_items(pdf)

        for item in items:
            item.update({
                "Invoice Date": inv_date,
                "Invoice Number": inv_no,
                "Invoice P.O.#": po_no,
                "Invoice Bill To": bill_to,
                "Invoice Ship To": ship_to,
                "Invoice Dispatch From": dispatch_from
            })
            all_data.append(item)

# =======================================================
# 8. FINAL DATAFRAME
# =======================================================
invoice_df = pd.DataFrame(all_data).drop_duplicates()
invoice_df = invoice_df[invoice_df["Invoice Item & Description"] != ""]

cols = [
    "Invoice Date", "Invoice Number", "Invoice P.O.#",
    "Invoice Bill To", "Invoice Ship To", "Invoice Dispatch From",
    "Invoice S.No", "Invoice Item & Description", "Invoice SKU No",
    "Invoice HSN/SAC", "Invoice Case Qty", "Invoice Rate",
    "IGST %", "IGST Amount",
    "CGST %", "CGST Amount",
    "SGST %", "SGST Amount",
    "Invoice Amount"
]

invoice_df = invoice_df[cols]

# =======================================================
# 9. SAVE OUTPUT
# =======================================================
output_file = os.path.join(OUTPUT_DIR, "FLIPKART_Invoice_With_GST.xlsx")
invoice_df.to_excel(output_file, index=False)

print("\n🎉 COMPLETED SUCCESSFULLY")
print("📁 Saved at:", output_file)
print("📊 Total rows:", len(invoice_df))


📂 Total PDFs: 262
📄 Processing: 20250409-70-huj9sy20241226004444234_54.pdf
📄 Processing: ESHBT261.pdf


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\3836131085.py:82: DeprecationWarning: 'maxsplit' is passed as positional argument
  parts = re.split(r'(GSTIN \w+)', full_addr, 1)


📄 Processing: ESHBT26105.pdf
📄 Processing: ESHBT26106.pdf
📄 Processing: ESHBT26107.pdf
📄 Processing: ESHBT26139.pdf
📄 Processing: ESHBT26171.pdf
📄 Processing: ESHBT262.pdf
📄 Processing: ESHBT263.pdf
📄 Processing: ESHBT264.pdf
📄 Processing: ESHBT265.pdf
📄 Processing: ESHBT266.pdf
📄 Processing: ESHBT267.pdf
📄 Processing: ESHM26103.pdf
📄 Processing: ESHM26123.pdf
📄 Processing: ESHM2625.pdf
📄 Processing: ESHM2626.pdf
📄 Processing: ESHM2627.pdf
📄 Processing: ESHM2628.pdf
📄 Processing: ESHM263.pdf
📄 Processing: ESHM264.pdf
📄 Processing: ESHM265.pdf
📄 Processing: ESHM266.pdf
📄 Processing: ESHM2669.pdf
📄 Processing: ESHT2645.pdf
📄 Processing: ESHU2612.pdf
📄 Processing: ESHU2613.pdf
📄 Processing: ESHU2614.pdf
📄 Processing: ESHU26165.pdf
📄 Processing: ESHU26166.pdf
📄 Processing: ESHU26167.pdf
📄 Processing: ESHU26168.pdf
📄 Processing: ESHU26169.pdf
📄 Processing: ESHU2627.pdf
📄 Processing: ESHU2628.pdf
📄 Processing: ESHU26288.pdf
📄 Processing: ESHU26289.pdf
📄 Processing: ESHU2629.pdf
📄 Processing:

In [8]:
invoice_df

,Invoice Date,Invoice Number,Invoice P.O.#,Invoice Bill To,Invoice Ship To,Invoice Dispatch From,Invoice S.No,Invoice Item & Description,Invoice SKU No,Invoice HSN/SAC,Invoice Case Qty,Invoice Rate,IGST %,IGST Amount,CGST %,CGST Amount,SGST %,SGST Amount,Invoice Amount
0,05/04/2026,ESHBT/26/1,FBHWN07868785,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,ZB- BS50,44219990,200.00,57.142,,,2.5%,285.71,2.5%,285.71,"11,428.40"
1,29/04/2026,ESHBT/26/105,FBHWN07976158,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,140mm Birchwood Spoon - - Count of 50 - Beige ...,ZB- BS140 MM50,44219990,200.00,42.857,,,2.5%,214.29,2.5%,214.29,"8,571.40"
2,29/04/2026,ESHBT/26/105,FBHWN07976158,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,120.00,52.796,,,9%,570.20,9%,570.20,"6,335.52"
3,29/04/2026,ESHBT/26/106,FBHWN08023669,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,320.00,52.796,,,9%,"1,520.52",9%,"1,520.52","16,894.72"
4,29/04/2026,ESHBT/26/107,FBHWN08043741,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,520.00,52.796,,,9%,"2,470.85",9%,"2,470.85","27,453.92"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380,25/03/2026,FK/2025-26/168,FNNWN07852010,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# :...",,Compostable Bagasse Bowl 180ml Round set of 25...,FG- CBB6RO2 5UB,48237010,64.00,48.90,5%,156.48,,,,,"3,129.60"
381,25/03/2026,FK/2025-26/168,FNNWN07852010,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# :...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ2 5NL,48236900,40.00,44.07,18%,317.30,,,,,"1,762.80"
382,25/03/2026,FK/2025-26/169,FNNWN07852203,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# :...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,ZB-BS50,44219990,100.00,57.14,5%,285.70,,,,,"5,714.00"
383,25/03/2026,FK/2025-26/170,FNNWN07860786,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# :...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ2 5NL,48236900,40.00,44.07,18%,317.30,,,,,"1,762.80"


# Filter out 2024 year

In [9]:
invoice_df['Invoice Date'] = pd.to_datetime(
    invoice_df['Invoice Date'],
    format='%d/%m/%Y',
    errors='coerce'
)
invoice_df['Invoice Date'] = (
    pd.to_datetime(
        invoice_df['Invoice Date'],
        dayfirst=True,
        errors='coerce'
    )
    .dt.normalize()
)

# Now this WILL work
invoice_df = invoice_df[invoice_df['Invoice Date'].dt.year != 2024]


# Save a dumped File

In [10]:
invoice_df

,Invoice Date,Invoice Number,Invoice P.O.#,Invoice Bill To,Invoice Ship To,Invoice Dispatch From,Invoice S.No,Invoice Item & Description,Invoice SKU No,Invoice HSN/SAC,Invoice Case Qty,Invoice Rate,IGST %,IGST Amount,CGST %,CGST Amount,SGST %,SGST Amount,Invoice Amount
0,2026-04-05,ESHBT/26/1,FBHWN07868785,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,ZB- BS50,44219990,200.00,57.142,,,2.5%,285.71,2.5%,285.71,"11,428.40"
1,2026-04-29,ESHBT/26/105,FBHWN07976158,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,140mm Birchwood Spoon - - Count of 50 - Beige ...,ZB- BS140 MM50,44219990,200.00,42.857,,,2.5%,214.29,2.5%,214.29,"8,571.40"
2,2026-04-29,ESHBT/26/105,FBHWN07976158,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,120.00,52.796,,,9%,570.20,9%,570.20,"6,335.52"
3,2026-04-29,ESHBT/26/106,FBHWN08023669,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,320.00,52.796,,,9%,"1,520.52",9%,"1,520.52","16,894.72"
4,2026-04-29,ESHBT/26/107,FBHWN08043741,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,520.00,52.796,,,9%,"2,470.85",9%,"2,470.85","27,453.92"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380,2026-03-25,FK/2025-26/168,FNNWN07852010,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# :...",,Compostable Bagasse Bowl 180ml Round set of 25...,FG- CBB6RO2 5UB,48237010,64.00,48.90,5%,156.48,,,,,"3,129.60"
381,2026-03-25,FK/2025-26/168,FNNWN07852010,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# :...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ2 5NL,48236900,40.00,44.07,18%,317.30,,,,,"1,762.80"
382,2026-03-25,FK/2025-26/169,FNNWN07852203,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# :...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,ZB-BS50,44219990,100.00,57.14,5%,285.70,,,,,"5,714.00"
383,2026-03-25,FK/2025-26/170,FNNWN07860786,FLIPKART INDIA PVT LTD Flipkart India Private ...,FLIPKART INDIA PVT LTD Flipkart India Private ...,": EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# :...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ2 5NL,48236900,40.00,44.07,18%,317.30,,,,,"1,762.80"


In [11]:
def clean_nan_words(x):
    if isinstance(x, str):
        # remove standalone 'nan' (case-insensitive)
        x = re.sub(r"\bFlipkart\b", "", x, flags=re.IGNORECASE)
        # normalize spaces
        x = re.sub(r"\s+", " ", x).strip()
    return x

def clean_colon_words(x):
    if isinstance(x, str):
        # remove colon only if it appears at the start (with or without spaces)
        x = re.sub(r"^\s*:\s*", "", x)

        # normalize spaces
        x = re.sub(r"\s+", " ", x).strip()
    return x

invoice_df["Invoice Ship To"] = invoice_df["Invoice Ship To"].apply(clean_nan_words)
invoice_df["Invoice Bill To"] = invoice_df["Invoice Bill To"].apply(clean_nan_words)
invoice_df["Invoice Dispatch From"] = invoice_df["Invoice Dispatch From"].apply(clean_colon_words)


In [12]:
invoice_df

,Invoice Date,Invoice Number,Invoice P.O.#,Invoice Bill To,Invoice Ship To,Invoice Dispatch From,Invoice S.No,Invoice Item & Description,Invoice SKU No,Invoice HSN/SAC,Invoice Case Qty,Invoice Rate,IGST %,IGST Amount,CGST %,CGST Amount,SGST %,SGST Amount,Invoice Amount
0,2026-04-05,ESHBT/26/1,FBHWN07868785,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,ZB- BS50,44219990,200.00,57.142,,,2.5%,285.71,2.5%,285.71,"11,428.40"
1,2026-04-29,ESHBT/26/105,FBHWN07976158,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,140mm Birchwood Spoon - - Count of 50 - Beige ...,ZB- BS140 MM50,44219990,200.00,42.857,,,2.5%,214.29,2.5%,214.29,"8,571.40"
2,2026-04-29,ESHBT/26/105,FBHWN07976158,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,120.00,52.796,,,9%,570.20,9%,570.20,"6,335.52"
3,2026-04-29,ESHBT/26/106,FBHWN08023669,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,320.00,52.796,,,9%,"1,520.52",9%,"1,520.52","16,894.72"
4,2026-04-29,ESHBT/26/107,FBHWN08043741,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,520.00,52.796,,,9%,"2,470.85",9%,"2,470.85","27,453.92"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380,2026-03-25,FK/2025-26/168,FNNWN07852010,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,Compostable Bagasse Bowl 180ml Round set of 25...,FG- CBB6RO2 5UB,48237010,64.00,48.90,5%,156.48,,,,,"3,129.60"
381,2026-03-25,FK/2025-26/168,FNNWN07852010,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ2 5NL,48236900,40.00,44.07,18%,317.30,,,,,"1,762.80"
382,2026-03-25,FK/2025-26/169,FNNWN07852203,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,ZB-BS50,44219990,100.00,57.14,5%,285.70,,,,,"5,714.00"
383,2026-03-25,FK/2025-26/170,FNNWN07860786,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ2 5NL,48236900,40.00,44.07,18%,317.30,,,,,"1,762.80"


In [13]:
po_df=final_df.copy()

In [14]:
po_df['FSN/ISBN13'].nunique()

30

# Use helper file to get SKU details for PO's

In [15]:
import pandas as pd

helper_path = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\helper_file.xlsx"

helper_df = pd.read_excel(helper_path)
po_df["FSN/ISBN13"] = po_df["FSN/ISBN13"].astype(str).str.strip()
helper_df["Item Code"] = helper_df["Item Code"].astype(str).str.strip()
helper_df = helper_df[
    ["Item Code", "SKU", "Material", "Category", "Sub Category","Box/Case"]
]
po_df = po_df.merge(
    helper_df,
    how="left",
    left_on="FSN/ISBN13",
    right_on="Item Code"
)
po_df.drop(columns=["Item Code"], inplace=True)
missing_fsn = (
    po_df.loc[po_df["SKU"].isna(), "FSN/ISBN13"]
    .dropna()
    .unique()
    .tolist()
)

print(f"❌ Total FSN/ISBN13 not found: {len(missing_fsn)}")


❌ Total FSN/ISBN13 not found: 1


In [16]:
missing_fsn

['CNSH47JACCHHYHZX']

In [17]:
po_df

,S. no.,HSN/SA Code,FSN/ISBN13,Quantity,Pending Quantity,UOM,Title,Required by Date,Supplier MRP,Supplier Price,Taxable Value,IGST Rate,IGST Amount(per unit),SGST Rate,SGST Amount(per unit),CGST Rate,CGST Amount(per unit),CESS Rate,CESS Amount(per unit),Tax Amount,Total Amount,PO Number,PO Date,PO Expiry,PO Delivered By Address,PO Delivered To,SKU,Material,Category,Sub Category,Box/Case
0,1,48237010.0,BWLH5FZYW9XZEX9F,192,192.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-10,119.0 INR,51.35 INR,9390.720000000001 INR,0.0%,0.0 INR,2.5%,,2.5%,1.22 INR,0.0%,0.0 INR,468.48 INR,9859.20,fagwn07808591,2026-03-11,10-04-26,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Survey No 518,...",CBB6RO25UB,Bagasse,Home Essentials,Kitchen Essentials,64.0
1,1,48237010.0,BWLH5FZYW9XZEX9F,128,128.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-23,119.0 INR,48.3 INR,5888.0 INR,0.0%,0.0 INR,2.5%,,2.5%,1.15 INR,0.0%,0.0 INR,294.4 INR,6182.40,fagwn07934808,2026-04-01,23-04-26,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Survey No 518,...",CBB6RO25UB,Bagasse,Home Essentials,Kitchen Essentials,64.0
2,1,48237010,BWLH5FZYW9XZEX9F,12,12.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-12,119.0 INR,48.3 INR,551.9999999999999 INR,5.0%,2.3 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,27.599999999999998 INR,579.60,faswn07894271,2026-03-26,13-04-26,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust...",CBB6RO25UB,Bagasse,Home Essentials,Kitchen Essentials,64.0
3,2,48237010,BWLH4H8YSGDZZX78,24,24.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-04-12,89.0 INR,35.0 INR,799.92 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,40.08 INR,840.00,faswn07894271,2026-03-26,13-04-26,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust...",PLB5RO10,Palm Leaf,Home Essentials,Tableware,20.0
4,3,46021919aa,PTDGXQBHGS5QPZZQ,12,12.0,pcs,ECO SOUL Square Palm Leaf 10 inch 10 Dinner Plate,2026-04-12,199.0 INR,118.3 INR,1419.6 INR,0.0%,0.0 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,0.0 INR,1419.60,faswn07894271,2026-03-26,13-04-26,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust...",PLP10SQ10,Palm Leaf,Home Essentials,Tableware,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,2,48236900.0,GLSH6KK67APTENYF,200,200.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-22,129.0 INR,62.3 INR,10560.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,1900.0 INR,12460.00,fuswn08080464,2026-04-29,22-05-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin...",PCL8OZ25NL,Paper,Home Essentials,Kitchen Essentials,40.0
940,1,48236900.0,GLSH6KK67APTENYF,320,320.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-24,129.0 INR,62.3 INR,16896.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,3040.0 INR,19936.00,fuswn08095377,2026-05-01,25-05-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin...",PCL8OZ25NL,Paper,Home Essentials,Kitchen Essentials,40.0
941,1,48237010.0,BWLH4H8YSGDZZX78,80,80.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-05-22,89.0 INR,35.0 INR,2666.4 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,133.6 INR,2800.00,fuswn08122617,2026-05-07,22-05-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin...",PLB5RO10,Palm Leaf,Home Essentials,Tableware,20.0
942,1,44191900.0,SPNH46S4G7HG3FWA,200,200.0,pcs,ECO SOUL_140 mm Biodegradable_Soup Spoon_Beige...,2026-07-04,99.0 INR,45.0 INR,8572.0 INR,5.0%,2.14 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,428.0 INR,9000.00,fuswn08276251,2026-06-10,06-07-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, E

# Invoice file cleaning

In [18]:
invoice_df['Invoice P.O.#']=invoice_df['Invoice P.O.#'].str.lower()

In [19]:
invoice_df

,Invoice Date,Invoice Number,Invoice P.O.#,Invoice Bill To,Invoice Ship To,Invoice Dispatch From,Invoice S.No,Invoice Item & Description,Invoice SKU No,Invoice HSN/SAC,Invoice Case Qty,Invoice Rate,IGST %,IGST Amount,CGST %,CGST Amount,SGST %,SGST Amount,Invoice Amount
0,2026-04-05,ESHBT/26/1,fbhwn07868785,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,ZB- BS50,44219990,200.00,57.142,,,2.5%,285.71,2.5%,285.71,"11,428.40"
1,2026-04-29,ESHBT/26/105,fbhwn07976158,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,140mm Birchwood Spoon - - Count of 50 - Beige ...,ZB- BS140 MM50,44219990,200.00,42.857,,,2.5%,214.29,2.5%,214.29,"8,571.40"
2,2026-04-29,ESHBT/26/105,fbhwn07976158,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,120.00,52.796,,,9%,570.20,9%,570.20,"6,335.52"
3,2026-04-29,ESHBT/26/106,fbhwn08023669,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,320.00,52.796,,,9%,"1,520.52",9%,"1,520.52","16,894.72"
4,2026-04-29,ESHBT/26/107,fbhwn08043741,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ 25NL,48236900,520.00,52.796,,,9%,"2,470.85",9%,"2,470.85","27,453.92"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380,2026-03-25,FK/2025-26/168,fnnwn07852010,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,Compostable Bagasse Bowl 180ml Round set of 25...,FG- CBB6RO2 5UB,48237010,64.00,48.90,5%,156.48,,,,,"3,129.60"
381,2026-03-25,FK/2025-26/168,fnnwn07852010,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ2 5NL,48236900,40.00,44.07,18%,317.30,,,,,"1,762.80"
382,2026-03-25,FK/2025-26/169,fnnwn07852203,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,ZB-BS50,44219990,100.00,57.14,5%,285.70,,,,,"5,714.00"
383,2026-03-25,FK/2025-26/170,fnnwn07860786,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,ZB- PCL8OZ2 5NL,48236900,40.00,44.07,18%,317.30,,,,,"1,762.80"


# Check the duplicacy of SKU and PO Number

In [20]:
is_unique = ~po_df.duplicated(subset=["SKU", "PO Number"]).any()
print("Is (SKU + PO_quantity) unique?", is_unique)

Is (SKU + PO_quantity) unique? True


In [21]:
po_df.drop_duplicates(subset=['SKU', 'PO Number'], inplace=True)


# Check the duplicacy of SKU and Invoice Number

In [22]:
is_unique = ~invoice_df.duplicated(subset=["Invoice SKU No", "Invoice P.O.#"]).any()
print("Is (SKU + PO_quantity) unique?", is_unique)

Is (SKU + PO_quantity) unique? True


In [23]:
invoice_df.drop_duplicates(subset=['Invoice SKU No', 'Invoice P.O.#'], inplace=True)


In [24]:
invoice_df.columns

Index(['Invoice Date', 'Invoice Number', 'Invoice P.O.#', 'Invoice Bill To',
       'Invoice Ship To', 'Invoice Dispatch From', 'Invoice S.No',
       'Invoice Item & Description', 'Invoice SKU No', 'Invoice HSN/SAC',
       'Invoice Case Qty', 'Invoice Rate', 'IGST %', 'IGST Amount', 'CGST %',
       'CGST Amount', 'SGST %', 'SGST Amount', 'Invoice Amount'],
      dtype='object')

# Rename CGST, IGST, SGST

In [25]:
invoice_df = invoice_df.rename(columns={
    'CGST Amount': 'CGST Amt',
    'SGST Amount': 'SGST Amt',
    'IGST Amount': 'IGST Amt',
    'Invoice Bill To':'invoice_bill_to',
    'Invoice Ship To':'invoice_ship_to',
    'Invoice Dispatch From':'invoice_dispatch_from'
})


# Calculate Invoice amount Inc GST

In [26]:
import pandas as pd

# Columns involved in total calculation
gst_cols = ["Invoice Amount", "CGST Amt", "SGST Amt", "IGST Amt"]

# Step 1: Clean + convert object columns to numeric
for col in gst_cols:
    invoice_df[col] = (
        invoice_df[col]
        .astype(str)                      # ensure string
        .str.replace(",", "", regex=False)  # remove thousand separators
        .str.strip()
    )
    invoice_df[col] = pd.to_numeric(
        invoice_df[col],
        errors="coerce"
    ).fillna(0)

# Step 2: Create invoice amount including GST
invoice_df["invoice_amt_gst"] = (
    invoice_df["Invoice Amount"]
    + invoice_df["CGST Amt"]
    + invoice_df["SGST Amt"]
    + invoice_df["IGST Amt"]
)

# Optional: round to 2 decimals (recommended for invoices)
invoice_df["invoice_amt_gst"] = invoice_df["invoice_amt_gst"].round(2)

# Preview result
invoice_df[
    ["Invoice Amount", "CGST Amt", "SGST Amt", "IGST Amt", "invoice_amt_gst"]
].head()


,Invoice Amount,CGST Amt,SGST Amt,IGST Amt,invoice_amt_gst
0,11428.40,285.71,285.71,0.0,11999.82
1,8571.40,214.29,214.29,0.0,8999.98
2,6335.52,570.20,570.20,0.0,7475.92
3,16894.72,1520.52,1520.52,0.0,19935.76
4,27453.92,2470.85,2470.85,0.0,32395.62


In [27]:
invoice_df['invoice_amt_gst'].sum()

np.float64(2797994.6500000004)

# Rename columns in PO

In [28]:
po_df = po_df.rename(columns={
    'PO Number': 'PO No',
    'PO Expiry': 'PO Expiry Date',
    'Quantity': 'PO_quantity',
    'Total Amount': 'PO_Amount',
    'HSN/SA Code': 'HSN Code'
})


# Rename columns in invoice

In [29]:
invoice_df = invoice_df.rename(columns={
    'Invoice SKU No': 'SKU',
    'Invoice Case Qty': 'Invoice_Quantity',
    'Invoice Amount': 'Invoice_Amount',
    'Invoice P.O.#': 'PO No'
})


# SKU cleaning in Invoice

In [30]:
import re
import pandas as pd

def clean_sku(s):
    if pd.isna(s):
        return ""

    s = str(s).upper().strip()

    # 1. Remove starting prefixes like "ZB -" or "SP -"
    s = re.sub(r"^(ZB|SP)\s*-\s*", "", s)

    # 2. Remove spaces
    s = s.replace(" ", "")

    # 3. Remove hyphens anywhere
    s = s.replace("-", "")

    # 4. Remove trailing dot
    s = re.sub(r"\.$", "", s)

    # 5. Keep only A–Z and 0–9
    s = re.sub(r"[^A-Z0-9]", "", s)

    return s


invoice_df['SKU'] = invoice_df['SKU'].apply(clean_sku)

In [31]:
po_df.columns

Index(['S. no.', 'HSN Code', 'FSN/ISBN13', 'PO_quantity', 'Pending Quantity',
       'UOM', 'Title', 'Required by Date', 'Supplier MRP', 'Supplier Price',
       'Taxable Value', 'IGST Rate', 'IGST Amount(per unit)', 'SGST Rate',
       'SGST Amount(per unit)', 'CGST Rate', 'CGST Amount(per unit)',
       'CESS Rate', 'CESS Amount(per unit)', 'Tax Amount', 'PO_Amount',
       'PO No', 'PO Date', 'PO Expiry Date', 'PO Delivered By Address',
       'PO Delivered To', 'SKU', 'Material', 'Category', 'Sub Category',
       'Box/Case'],
      dtype='object')

In [32]:
invoice_df.columns

Index(['Invoice Date', 'Invoice Number', 'PO No', 'invoice_bill_to',
       'invoice_ship_to', 'invoice_dispatch_from', 'Invoice S.No',
       'Invoice Item & Description', 'SKU', 'Invoice HSN/SAC',
       'Invoice_Quantity', 'Invoice Rate', 'IGST %', 'IGST Amt', 'CGST %',
       'CGST Amt', 'SGST %', 'SGST Amt', 'Invoice_Amount', 'invoice_amt_gst'],
      dtype='object')

In [33]:
import pandas as pd

# Convert 'PO_Amount' from object to numeric safely
po_df['PO_Amount'] = (
    po_df['PO_Amount']
    .astype(str)
    .str.replace(r'[\s,]+', '', regex=True)  # remove spaces, \n, and commas
    .astype(float)
)

# Optional: round to 2 decimals
po_df["PO_Amount"] = po_df["PO_Amount"].round(2)

# Verify datatype
po_df["PO_Amount"].dtype


dtype('float64')

In [34]:
po_df['PO_Amount'].sum()

np.float64(5876417.75)

# Merge PO Invoice 

In [35]:
merged_df = po_df.merge(
    invoice_df,
    on=['PO No', 'SKU'],
    how='left'   # use 'inner' if you want only matched records
)


In [36]:
invoice_df

,Invoice Date,Invoice Number,PO No,invoice_bill_to,invoice_ship_to,invoice_dispatch_from,Invoice S.No,Invoice Item & Description,SKU,Invoice HSN/SAC,Invoice_Quantity,Invoice Rate,IGST %,IGST Amt,CGST %,CGST Amt,SGST %,SGST Amt,Invoice_Amount,invoice_amt_gst
0,2026-04-05,ESHBT/26/1,fbhwn07868785,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,BS50,44219990,200.00,57.142,,0.00,2.5%,285.71,2.5%,285.71,11428.40,11999.82
1,2026-04-29,ESHBT/26/105,fbhwn07976158,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,140mm Birchwood Spoon - - Count of 50 - Beige ...,BS140MM50,44219990,200.00,42.857,,0.00,2.5%,214.29,2.5%,214.29,8571.40,8999.98
2,2026-04-29,ESHBT/26/105,fbhwn07976158,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,PCL8OZ25NL,48236900,120.00,52.796,,0.00,9%,570.20,9%,570.20,6335.52,7475.92
3,2026-04-29,ESHBT/26/106,fbhwn08023669,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,PCL8OZ25NL,48236900,320.00,52.796,,0.00,9%,1520.52,9%,1520.52,16894.72,19935.76
4,2026-04-29,ESHBT/26/107,fbhwn08043741,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"Ecosoul Home Pvt Ltd 1st Phase, Plot No. P.O.#...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,PCL8OZ25NL,48236900,520.00,52.796,,0.00,9%,2470.85,9%,2470.85,27453.92,32395.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380,2026-03-25,FK/2025-26/168,fnnwn07852010,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,Compostable Bagasse Bowl 180ml Round set of 25...,FGCBB6RO25UB,48237010,64.00,48.90,5%,156.48,,0.00,,0.00,3129.60,3286.08
381,2026-03-25,FK/2025-26/168,fnnwn07852010,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,PCL8OZ25NL,48236900,40.00,44.07,18%,317.30,,0.00,,0.00,1762.80,2080.10
382,2026-03-25,FK/2025-26/169,fnnwn07852203,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,160mm Birchwood Spoon - - Count of 50 - Beige ...,BS50,44219990,100.00,57.14,5%,285.70,,0.00,,0.00,5714.00,5999.70
383,2026-03-25,FK/2025-26/170,fnnwn07860786,INDIA PVT LTD India Private Limited India Priv...,INDIA PVT LTD India Private Limited India Priv...,"EcoSoul Home Pvt Ltd Plot 305, Udyog P.O.# : F...",,Compostable Paper Cups 8 OZ - Set Of 25 NL SKU...,PCL8OZ25NL,48236900,40.00,44.07,18%,317.30,,0.00,,0.00,1762.80,2080.10


In [37]:
po_df

,S. no.,HSN Code,FSN/ISBN13,PO_quantity,Pending Quantity,UOM,Title,Required by Date,Supplier MRP,Supplier Price,Taxable Value,IGST Rate,IGST Amount(per unit),SGST Rate,SGST Amount(per unit),CGST Rate,CGST Amount(per unit),CESS Rate,CESS Amount(per unit),Tax Amount,PO_Amount,PO No,PO Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,SKU,Material,Category,Sub Category,Box/Case
0,1,48237010.0,BWLH5FZYW9XZEX9F,192,192.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-10,119.0 INR,51.35 INR,9390.720000000001 INR,0.0%,0.0 INR,2.5%,,2.5%,1.22 INR,0.0%,0.0 INR,468.48 INR,9859.2,fagwn07808591,2026-03-11,10-04-26,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Survey No 518,...",CBB6RO25UB,Bagasse,Home Essentials,Kitchen Essentials,64.0
1,1,48237010.0,BWLH5FZYW9XZEX9F,128,128.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-23,119.0 INR,48.3 INR,5888.0 INR,0.0%,0.0 INR,2.5%,,2.5%,1.15 INR,0.0%,0.0 INR,294.4 INR,6182.4,fagwn07934808,2026-04-01,23-04-26,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Survey No 518,...",CBB6RO25UB,Bagasse,Home Essentials,Kitchen Essentials,64.0
2,1,48237010,BWLH5FZYW9XZEX9F,12,12.0,pcs,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,2026-04-12,119.0 INR,48.3 INR,551.9999999999999 INR,5.0%,2.3 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,27.599999999999998 INR,579.6,faswn07894271,2026-03-26,13-04-26,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust...",CBB6RO25UB,Bagasse,Home Essentials,Kitchen Essentials,64.0
3,2,48237010,BWLH4H8YSGDZZX78,24,24.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-04-12,89.0 INR,35.0 INR,799.92 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,40.08 INR,840.0,faswn07894271,2026-03-26,13-04-26,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust...",PLB5RO10,Palm Leaf,Home Essentials,Tableware,20.0
4,3,46021919aa,PTDGXQBHGS5QPZZQ,12,12.0,pcs,ECO SOUL Square Palm Leaf 10 inch 10 Dinner Plate,2026-04-12,199.0 INR,118.3 INR,1419.6 INR,0.0%,0.0 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,0.0 INR,1419.6,faswn07894271,2026-03-26,13-04-26,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust...",PLP10SQ10,Palm Leaf,Home Essentials,Tableware,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,2,48236900.0,GLSH6KK67APTENYF,200,200.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-22,129.0 INR,62.3 INR,10560.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,1900.0 INR,12460.0,fuswn08080464,2026-04-29,22-05-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin...",PCL8OZ25NL,Paper,Home Essentials,Kitchen Essentials,40.0
940,1,48236900.0,GLSH6KK67APTENYF,320,320.0,pcs,ECO SOUL_Glass Set_White_810103156647_25,2026-05-24,129.0 INR,62.3 INR,16896.0 INR,18.0%,9.5 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,3040.0 INR,19936.0,fuswn08095377,2026-05-01,25-05-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin...",PCL8OZ25NL,Paper,Home Essentials,Kitchen Essentials,40.0
941,1,48237010.0,BWLH4H8YSGDZZX78,80,80.0,pcs,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,2026-05-22,89.0 INR,35.0 INR,2666.4 INR,5.0%,1.67 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,133.6 INR,2800.0,fuswn08122617,2026-05-07,22-05-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin...",PLB5RO10,Palm Leaf,Home Essentials,Tableware,20.0
942,1,44191900.0,SPNH46S4G7HG3FWA,200,200.0,pcs,ECO SOUL_140 mm Biodegradable_Soup Spoon_Beige...,2026-07-04,99.0 INR,45.0 INR,8572.0 INR,5.0%,2.14 INR,0.0%,,0.0%,0.0 INR,0.0%,0.0 INR,428.0 INR,9000.0,fuswn08276251,2026-06-10,06-07-26,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehous

In [38]:
merged_df['invoice_amt_gst'].sum()

np.float64(2645358.23)

In [39]:
merged_df.columns

Index(['S. no.', 'HSN Code', 'FSN/ISBN13', 'PO_quantity', 'Pending Quantity',
       'UOM', 'Title', 'Required by Date', 'Supplier MRP', 'Supplier Price',
       'Taxable Value', 'IGST Rate', 'IGST Amount(per unit)', 'SGST Rate',
       'SGST Amount(per unit)', 'CGST Rate', 'CGST Amount(per unit)',
       'CESS Rate', 'CESS Amount(per unit)', 'Tax Amount', 'PO_Amount',
       'PO No', 'PO Date', 'PO Expiry Date', 'PO Delivered By Address',
       'PO Delivered To', 'SKU', 'Material', 'Category', 'Sub Category',
       'Box/Case', 'Invoice Date', 'Invoice Number', 'invoice_bill_to',
       'invoice_ship_to', 'invoice_dispatch_from', 'Invoice S.No',
       'Invoice Item & Description', 'Invoice HSN/SAC', 'Invoice_Quantity',
       'Invoice Rate', 'IGST %', 'IGST Amt', 'CGST %', 'CGST Amt', 'SGST %',
       'SGST Amt', 'Invoice_Amount', 'invoice_amt_gst'],
      dtype='object')

# Take required columns

In [40]:
required_columns = [
    'FSN/ISBN13',
    'HSN Code',
    'PO No',
    'PO Date',
    'Title',
    'PO Expiry Date',
    'PO Delivered By Address',
    'PO Delivered To',
    'Invoice Number',
    'Supplier Price',
    'invoice_bill_to',
    'invoice_ship_to',
    'invoice_dispatch_from',
    'Tax Amount',
    'Invoice Date',
    'Supplier MRP',
    'SKU',
    'Category',
    'Material',
    'Sub Category',
    'PO_quantity',
    'Box/Case',
    'Invoice_Quantity',
    'PO_Amount',
    'Invoice_Amount',
    'invoice_amt_gst',
    'SGST Amt',
    'IGST Amt',
    'CGST Amt',
    'Invoice Item & Description',
    'Invoice Rate',
    "IGST %",
    "CGST %",
    "SGST %"
]
existing_columns = [col for col in required_columns if col in merged_df.columns]
final_df = merged_df[existing_columns]


# Add status

In [41]:
import numpy as np
final_df['Invoice_Quantity'] = pd.to_numeric(
    final_df['Invoice_Quantity'], errors='coerce'
).fillna(0)

final_df['PO_quantity'] = pd.to_numeric(
    final_df['PO_quantity'], errors='coerce'
).fillna(0)

final_df['Status'] = np.where(
    final_df['Invoice Number'].isna(),
    'Unfulfilled',
    np.where(final_df['Invoice_Quantity'] < final_df['PO_quantity'], 'Short Shipped', 'Fulfilled')
)


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\1972213374.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['Invoice_Quantity'] = pd.to_numeric(
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\1972213374.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['PO_quantity'] = pd.to_numeric(
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\1972213374.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

# Correct the format of date

In [42]:
final_df = final_df.rename(columns={
    "PO Date": "PO Creation Date",
    "PO No": "PO Number"
})


# Add clientID, Platform, Channel

In [43]:
final_df[['ClientID', 'Platform', 'Channel']] = [
    'TH-1755734939046', 'QuickCommerce', 'Flikart']

In [44]:
final_df.columns

Index(['FSN/ISBN13', 'HSN Code', 'PO Number', 'PO Creation Date', 'Title',
       'PO Expiry Date', 'PO Delivered By Address', 'PO Delivered To',
       'Invoice Number', 'Supplier Price', 'invoice_bill_to',
       'invoice_ship_to', 'invoice_dispatch_from', 'Tax Amount',
       'Invoice Date', 'Supplier MRP', 'SKU', 'Category', 'Material',
       'Sub Category', 'PO_quantity', 'Box/Case', 'Invoice_Quantity',
       'PO_Amount', 'Invoice_Amount', 'invoice_amt_gst', 'SGST Amt',
       'IGST Amt', 'CGST Amt', 'Invoice Item & Description', 'Invoice Rate',
       'IGST %', 'CGST %', 'SGST %', 'Status', 'ClientID', 'Platform',
       'Channel'],
      dtype='object')

# Correct Data Types

In [45]:
import pandas as pd

# === 1. Convert date columns to datetime (date format) ===
date_cols = ["PO Creation Date", "PO Expiry Date"]

for col in date_cols:
    final_df[col] = pd.to_datetime(
        final_df[col],
        errors="coerce"   # invalid dates → NaT
    ).dt.date           # keep only date part


# === 2. Convert quantity & amount columns to integer ===
int_cols = [
    "PO_quantity",
    "Invoice_Quantity",
    "PO_Amount",
    "Invoice_Amount"
]

for col in int_cols:
    final_df[col] = (
        final_df[col]
        .astype(str)
        .str.replace(r"[^\d.]", "", regex=True)  # remove commas, currency, etc.
        .replace("", pd.NA)
    )

    final_df[col] = (
        pd.to_numeric(final_df[col], errors="coerce")  # 🔥 safest step
        .fillna(0)
        .astype(int)
    )



C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\2157732036.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  final_df[col] = pd.to_datetime(


In [46]:
final_df.columns

Index(['FSN/ISBN13', 'HSN Code', 'PO Number', 'PO Creation Date', 'Title',
       'PO Expiry Date', 'PO Delivered By Address', 'PO Delivered To',
       'Invoice Number', 'Supplier Price', 'invoice_bill_to',
       'invoice_ship_to', 'invoice_dispatch_from', 'Tax Amount',
       'Invoice Date', 'Supplier MRP', 'SKU', 'Category', 'Material',
       'Sub Category', 'PO_quantity', 'Box/Case', 'Invoice_Quantity',
       'PO_Amount', 'Invoice_Amount', 'invoice_amt_gst', 'SGST Amt',
       'IGST Amt', 'CGST Amt', 'Invoice Item & Description', 'Invoice Rate',
       'IGST %', 'CGST %', 'SGST %', 'Status', 'ClientID', 'Platform',
       'Channel'],
      dtype='object')

# Rename columns


In [47]:
final_df = final_df.rename(columns={
    "FSN/ISBN13":"PO Item Code",
    "Title":"PO Product Description",
    "Supplier Price":"PO Basic Cost Price",
    "Tax Amount":"PO Tax Amt",
    "Supplier MRP":"PO MRP",
})


In [48]:
final_df.columns

Index(['PO Item Code', 'HSN Code', 'PO Number', 'PO Creation Date',
       'PO Product Description', 'PO Expiry Date', 'PO Delivered By Address',
       'PO Delivered To', 'Invoice Number', 'PO Basic Cost Price',
       'invoice_bill_to', 'invoice_ship_to', 'invoice_dispatch_from',
       'PO Tax Amt', 'Invoice Date', 'PO MRP', 'SKU', 'Category', 'Material',
       'Sub Category', 'PO_quantity', 'Box/Case', 'Invoice_Quantity',
       'PO_Amount', 'Invoice_Amount', 'invoice_amt_gst', 'SGST Amt',
       'IGST Amt', 'CGST Amt', 'Invoice Item & Description', 'Invoice Rate',
       'IGST %', 'CGST %', 'SGST %', 'Status', 'ClientID', 'Platform',
       'Channel'],
      dtype='object')

In [49]:
merged_df=final_df.copy()

# Add NLC to calculate revenue

In [50]:
pricing_path = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Shambhavi's File\NLC All platform1.xlsx"



pricing_df = pd.read_excel(
    pricing_path,
    sheet_name="Flipkart"   # 🔥 THIS FIX
)


pricing_df = pricing_df[[
    "SKUs",
    "As per pricing sheetNLC",
    "MRP as per pricing sheet",
    "GST",
]].rename(columns={
    "SKUs": "SKU",
    "As per pricing sheetNLC":"NLC as per pricing sheet",
    "GST": "GST as per pricing sheet",
})
import re

def clean_sku(s):
    if pd.isna(s):
        return ""
    s = str(s).upper()
    s = re.sub(r"[^A-Z0-9]", "", s)
    return s
pricing_df["SKU_CLEAN"] = pricing_df["SKU"].apply(clean_sku)
merged_df["SKU_CLEAN"] = merged_df["SKU"].apply(clean_sku)
merged_df["SKU_CLEAN"] = merged_df["SKU"].apply(clean_sku)
merged_df = merged_df.merge(
    pricing_df,
    on="SKU_CLEAN",
    how="left"
)


In [51]:
pricing_df

,SKU,NLC as per pricing sheet,MRP as per pricing sheet,GST as per pricing sheet,SKU_CLEAN
0,BS140MM50,44.85,99,0.05,BS140MM50
1,BS50,64.35,149,0.05,BS50
2,BSF50,64.35,149,0.05,BSF50
3,BSP50,64.35,149,0.05,BSP50
4,CBB4SQ25UB,51.35,119,0.05,CBB4SQ25UB
5,CBB6RO25UB,51.35,119,0.05,CBB6RO25UB
6,CBDN1P100P01,38.35,89,0.18,CBDN1P100P01
7,CBP10RO25BL,213.00,399,0.00,CBP10RO25BL
8,CBP10RO3C25BL,213.00,399,0.00,CBP10RO3C25BL
9,CBP11RO4C10UB,90.35,199,0.05,CBP11RO4C10UB


# Calculate revenue

In [52]:
import numpy as np

# Ensure numeric types (very important)
merged_df['NLC as per pricing sheet'] = pd.to_numeric(merged_df['NLC as per pricing sheet'], errors='coerce')
merged_df['Invoice_Quantity'] = pd.to_numeric(merged_df['Invoice_Quantity'], errors='coerce')

# Create revenue column
merged_df['revenue'] = merged_df['NLC as per pricing sheet'] * merged_df['Invoice_Quantity']


# Drop unnecessary columns

In [53]:
merged_df.rename(columns={
    'SKU_x': 'SKU'
}, inplace=True)
merged_df.drop(columns='SKU_y', inplace=True)

columns_to_drop = [
    "SKU_CLEAN"
]

merged_df = merged_df.drop(columns=columns_to_drop, errors="ignore")

# Add Extra columns

In [54]:
extra_columns = [
    'grn_item_code',
    'PO Product UPC',
    'grn_product_description',
    'grn_mrp',
    'grn_tax_amount',
    'grn_quantity',
    'grn_total_amount',
    'grn_gmv_loss',
    'dn_date',
    'facility_name',
    'dn_id',
    'item_name',
    'vendor_invoice_id',
    'dn_quantity',
    'dn_value',
    'dn_reason',
    'dn_remark',
    'PO Landing Rate'
]

for col in extra_columns:
    merged_df[col] = ""


In [55]:
merged_df.columns

Index(['PO Item Code', 'HSN Code', 'PO Number', 'PO Creation Date',
       'PO Product Description', 'PO Expiry Date', 'PO Delivered By Address',
       'PO Delivered To', 'Invoice Number', 'PO Basic Cost Price',
       'invoice_bill_to', 'invoice_ship_to', 'invoice_dispatch_from',
       'PO Tax Amt', 'Invoice Date', 'PO MRP', 'SKU', 'Category', 'Material',
       'Sub Category', 'PO_quantity', 'Box/Case', 'Invoice_Quantity',
       'PO_Amount', 'Invoice_Amount', 'invoice_amt_gst', 'SGST Amt',
       'IGST Amt', 'CGST Amt', 'Invoice Item & Description', 'Invoice Rate',
       'IGST %', 'CGST %', 'SGST %', 'Status', 'ClientID', 'Platform',
       'Channel', 'NLC as per pricing sheet', 'MRP as per pricing sheet',
       'GST as per pricing sheet', 'revenue', 'grn_item_code',
       'PO Product UPC', 'grn_product_description', 'grn_mrp',
       'grn_tax_amount', 'grn_quantity', 'grn_total_amount', 'grn_gmv_loss',
       'dn_date', 'facility_name', 'dn_id', 'item_name', 'vendor_invoice_id

# Priorities the columns

In [56]:
priority_cols = [
    "PO Number",
    "PO Creation Date",
    "PO Expiry Date",
    "PO Delivered By Address",
    "PO Delivered To",
    "PO Item Code",
    "HSN Code",
    "PO Product UPC",
    "PO Product Description",
    "PO Basic Cost Price",
    "PO Tax Amt",
    "PO Landing Rate",
    "Invoice Number",
    "Invoice Date",
    "invoice_dispatch_from",
    "invoice_ship_to",
    "invoice_bill_to",
    "SKU",
    "Invoice Item & Description",
    "Category",
    "Material",
    "Sub Category",
    "PO_quantity",
    "Invoice_Quantity",
    "PO_Amount",
    "Invoice_Amount",
    "Invoice_amt_gst",
    "Status",
    "ClientID",
    "Platform",
    "Channel",
    "SGST %",
    "SGST Amt",
    "CGST %",
    "CGST Amt",
    "IGST %",
    "IGST Amt"
]
# Ensure we only use columns that actually exist
priority_cols_existing = [c for c in priority_cols if c in merged_df.columns]

# Remaining columns (everything else, original order preserved)
remaining_cols = [c for c in merged_df.columns if c not in priority_cols_existing]

# Final reordered dataframe
final_df = merged_df[priority_cols_existing + remaining_cols]


In [57]:
final_df.columns

Index(['PO Number', 'PO Creation Date', 'PO Expiry Date',
       'PO Delivered By Address', 'PO Delivered To', 'PO Item Code',
       'HSN Code', 'PO Product UPC', 'PO Product Description',
       'PO Basic Cost Price', 'PO Tax Amt', 'PO Landing Rate',
       'Invoice Number', 'Invoice Date', 'invoice_dispatch_from',
       'invoice_ship_to', 'invoice_bill_to', 'SKU',
       'Invoice Item & Description', 'Category', 'Material', 'Sub Category',
       'PO_quantity', 'Invoice_Quantity', 'PO_Amount', 'Invoice_Amount',
       'Status', 'ClientID', 'Platform', 'Channel', 'SGST %', 'SGST Amt',
       'CGST %', 'CGST Amt', 'IGST %', 'IGST Amt', 'PO MRP', 'Box/Case',
       'invoice_amt_gst', 'Invoice Rate', 'NLC as per pricing sheet',
       'MRP as per pricing sheet', 'GST as per pricing sheet', 'revenue',
       'grn_item_code', 'grn_product_description', 'grn_mrp', 'grn_tax_amount',
       'grn_quantity', 'grn_total_amount', 'grn_gmv_loss', 'dn_date',
       'facility_name', 'dn_id', 'item

# Add appointmemt date & status from file

In [58]:
import pandas as pd

# Load flipkart file
flipkart_file = r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Anoop Kumar's files - Flipkart Delivery Sheet\ESH - Flipkart Delivery Details.xlsx"
flipkart_df = pd.read_excel(flipkart_file)  # ← FIXED!


# Select only required columns and rename them
flipkart_selected = flipkart_df[[
    'PO Number', 
    'Appt Date', 
    'PO Status', 
    'Req Appt Date'
]].copy()
import pandas as pd

# Rename columns as requested
flipkart_selected = flipkart_selected.rename(columns={
    'Appt Date': 'appt date', 
    'PO Status': 'PO_fulfilled_status_from_file',
    'Req Appt Date': 'requested_appt_date'
})
flipkart_selected['PO Number'] = flipkart_selected['PO Number'].str.lower()
# Merge with final_df (fix column name mismatch)
final_df['PO Number'] = final_df['PO Number'].astype(str)
flipkart_selected['PO Number'] = flipkart_selected['PO Number'].astype(str)

final_df = final_df.merge(
    flipkart_selected,
    on='PO Number',
    how='left'  # Keeps all rows from final_df
)


print("✅ Merge complete!")
print(f"New columns: appt date, PO_fulfilled_status_from_file, requested_appt_date")


✅ Merge complete!
New columns: appt date, PO_fulfilled_status_from_file, requested_appt_date


In [59]:
final_df['PO_fulfilled_status_from_file'].isna().sum()

np.int64(567)

In [60]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,HSN Code,PO Product UPC,PO Product Description,PO Basic Cost Price,PO Tax Amt,PO Landing Rate,Invoice Number,Invoice Date,invoice_dispatch_from,invoice_ship_to,invoice_bill_to,SKU,Invoice Item & Description,Category,Material,Sub Category,PO_quantity,Invoice_Quantity,PO_Amount,Invoice_Amount,Status,ClientID,Platform,Channel,SGST %,SGST Amt,CGST %,CGST Amt,IGST %,IGST Amt,PO MRP,Box/Case,invoice_amt_gst,Invoice Rate,NLC as per pricing sheet,MRP as per pricing sheet,GST as per pricing sheet,revenue,grn_item_code,grn_product_description,grn_mrp,grn_tax_amount,grn_quantity,grn_total_amount,grn_gmv_loss,dn_date,facility_name,dn_id,item_name,vendor_invoice_id,dn_quantity,dn_value,dn_reason,dn_remark,appt date,PO_fulfilled_status_from_file,requested_appt_date
0,fagwn07808591,2026-03-11,2026-10-04,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Survey No 518,...",BWLH5FZYW9XZEX9F,48237010.0,,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,51.35 INR,468.48 INR,,NaN,NaT,NaN,NaN,NaN,CBB6RO25UB,NaN,Home Essentials,Bagasse,Kitchen Essentials,192,0,9859,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,119.0 INR,64.0,NaN,NaN,51.35,119.0,0.05,0.0,,,,,,,,,,,,,,,,,Cancelled by Vendor,Cancelled by Vendor,Cancelled by Vendor
1,fagwn07934808,2026-04-01,2026-04-23,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Survey No 518,...",BWLH5FZYW9XZEX9F,48237010.0,,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,48.3 INR,294.4 INR,,NaN,NaT,NaN,NaN,NaN,CBB6RO25UB,NaN,Home Essentials,Bagasse,Kitchen Essentials,128,0,6182,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,119.0 INR,64.0,NaN,NaN,51.35,119.0,0.05,0.0,,,,,,,,,,,,,,,,,Cancelled by Vendor,Cancelled by Vendor,Cancelled by Vendor
2,faswn07894271,2026-03-26,2026-04-13,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust...",BWLH5FZYW9XZEX9F,48237010,,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,48.3 INR,27.599999999999998 INR,,NaN,NaT,NaN,NaN,NaN,CBB6RO25UB,NaN,Home Essentials,Bagasse,Kitchen Essentials,12,0,579,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,119.0 INR,64.0,NaN,NaN,51.35,119.0,0.05,0.0,,,,,,,,,,,,,,,,,Cancelled by Vendor,Cancelled by Vendor,Cancelled by Vendor
3,faswn07894271,2026-03-26,2026-04-13,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust...",BWLH4H8YSGDZZX78,48237010,,ECO SOUL_Compostable Palm Leaf Bowls - Round 5...,35.0 INR,40.08 INR,,NaN,NaT,NaN,NaN,NaN,PLB5RO10,NaN,Home Essentials,Palm Leaf,Tableware,24,0,840,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,89.0 INR,20.0,NaN,NaN,38.35,89.0,0.00,0.0,,,,,,,,,,,,,,,,,Cancelled by Vendor,Cancelled by Vendor,Cancelled by Vendor
4,faswn07894271,2026-03-26,2026-04-13,"Ground Floor, 1, Pinnac Apartment Bamboo Gali,...","Flipkart India Private Limited, Hillway Indust...",PTDGXQBHGS5QPZZQ,46021919aa,,ECO SOUL Square Palm Leaf 10 inch 10 Dinner Plate,118.3 INR,0.0 INR,,NaN,NaT,NaN,NaN,NaN,PLP10SQ10,NaN,Home Essentials,Palm Leaf,Tableware,12,0,1419,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,199.0 INR,20.0,NaN,NaN,96.85,199.0,0.00,0.0,,,,,,,,,,,,,,,,,Cancelled by Vendor,Cancelled by Vendor,Cancelled by Vendor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,fuswn08080464,2026-04-29,2026-05-22,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin...",GLSH6KK67APTENYF,48236900.0,,ECO SOUL_Glass Set_White_810103156647_25,62.3 INR,1900.0 INR,,NaN,NaT,NaN,NaN,NaN,PCL8OZ25NL,NaN,H

In [61]:
final_df.columns

Index(['PO Number', 'PO Creation Date', 'PO Expiry Date',
       'PO Delivered By Address', 'PO Delivered To', 'PO Item Code',
       'HSN Code', 'PO Product UPC', 'PO Product Description',
       'PO Basic Cost Price', 'PO Tax Amt', 'PO Landing Rate',
       'Invoice Number', 'Invoice Date', 'invoice_dispatch_from',
       'invoice_ship_to', 'invoice_bill_to', 'SKU',
       'Invoice Item & Description', 'Category', 'Material', 'Sub Category',
       'PO_quantity', 'Invoice_Quantity', 'PO_Amount', 'Invoice_Amount',
       'Status', 'ClientID', 'Platform', 'Channel', 'SGST %', 'SGST Amt',
       'CGST %', 'CGST Amt', 'IGST %', 'IGST Amt', 'PO MRP', 'Box/Case',
       'invoice_amt_gst', 'Invoice Rate', 'NLC as per pricing sheet',
       'MRP as per pricing sheet', 'GST as per pricing sheet', 'revenue',
       'grn_item_code', 'grn_product_description', 'grn_mrp', 'grn_tax_amount',
       'grn_quantity', 'grn_total_amount', 'grn_gmv_loss', 'dn_date',
       'facility_name', 'dn_id', 'item

# Dataframe with refined values with status of did not delivered or cancelled by vendor

In [62]:
final_df['PO_fulfilled_status_from_file'].unique()

array(['Cancelled by Vendor', nan, 'Fulfilled', 'Expired',
       'Active-Action Required'], dtype=object)

In [63]:
final_df = final_df[~final_df['PO_fulfilled_status_from_file']
                    .isin(['Did not deliver', 'Cancelled by Vendor'])]


In [64]:
len(final_df.columns)

63

# Save the file to the destination

In [65]:
final_df.to_csv(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\flipkart_output.csv",index=False)

# Read the file

In [66]:
import pandas as pd
final_df.to_csv(r"C:\Users\Shubham Verma\OneDrive - EcoSoul Home\Data analytics_Ecosoul - Central Repository\E-commerce\India\India Retail- Quick Commerce - INDIA TEAM\quick_comm_final_output\flipkart_output.csv")

In [67]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,HSN Code,PO Product UPC,PO Product Description,PO Basic Cost Price,PO Tax Amt,PO Landing Rate,Invoice Number,Invoice Date,invoice_dispatch_from,invoice_ship_to,invoice_bill_to,SKU,Invoice Item & Description,Category,Material,Sub Category,PO_quantity,Invoice_Quantity,PO_Amount,Invoice_Amount,Status,ClientID,Platform,Channel,SGST %,SGST Amt,CGST %,CGST Amt,IGST %,IGST Amt,PO MRP,Box/Case,invoice_amt_gst,Invoice Rate,NLC as per pricing sheet,MRP as per pricing sheet,GST as per pricing sheet,revenue,grn_item_code,grn_product_description,grn_mrp,grn_tax_amount,grn_quantity,grn_total_amount,grn_gmv_loss,dn_date,facility_name,dn_id,item_name,vendor_invoice_id,dn_quantity,dn_value,dn_reason,dn_remark,appt date,PO_fulfilled_status_from_file,requested_appt_date
10,fbhwn05718319,2025-01-09,2025-01-28,"Tower-B, 2nd Floor, Unit-B 202A, Sector 142, ,...","Flipkart India Private Limited, Sy no 18/2, 18...",CNSH5F7WSMGXZFQX,48236900.0,,ECO SOUL_Cup Set_White,42.0 INR,282.04 INR,,NaN,NaT,NaN,NaN,NaN,PCL5OZ50NL,NaN,Home Essentials,Paper,Kitchen Essentials,44,0,1848,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,99.0 INR,40.0,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,,,,,,,,,NaN,NaN,NaN
11,fbhwn05769303,2025-01-21,2025-11-02,"Tower-B, 2nd Floor, Unit-B 202A, Sector 142, ,...","Flipkart India Private Limited, Sy no 18/2, 18...",CNSH5F7WSMGXZFQX,48236900.0,,ECO SOUL_Cup Set_White,42.0 INR,0.0 INR,,NaN,NaT,NaN,NaN,NaN,PCL5OZ50NL,NaN,Home Essentials,Paper,Kitchen Essentials,0,0,0,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,99.0 INR,40.0,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,,,,,,,,,NaN,NaN,NaN
12,fbhwn05849736,2025-02-11,2025-02-28,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Sy no 18/2, 18...",GLSH27KKZE99D6KH,48236900.0,,ECO SOUL_Glass_White_100,116.0 INR,3964.7999999999997 INR,,NaN,NaT,NaN,NaN,NaN,PCL5OZ100NL,NaN,Home Essentials,Paper,Tableware,224,0,25984,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,599.0 INR,20.0,NaN,NaN,116.00,199.0,0.18,0.0,,,,,,,,,,,,,,,,,NaN,NaN,NaN
13,fbhwn05849736,2025-02-11,2025-02-28,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Sy no 18/2, 18...",GLSH27KQJNDBMGYD,48236900.0,,ECO SOUL_Glass_White_50,116.0 INR,14160.0 INR,,NaN,NaT,NaN,NaN,NaN,PCL8OZ50NL,NaN,Home Essentials,Paper,Tableware,800,0,92800,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,549.0 INR,20.0,NaN,NaN,116.00,199.0,0.18,0.0,,,,,,,,,,,,,,,,,NaN,NaN,NaN
14,fbhwn05849736,2025-02-11,2025-02-28,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Sy no 18/2, 18...",BWLH5FZYW9XZEX9F,48237010.0,,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,51.35 INR,4576.0 INR,,NaN,NaT,NaN,NaN,NaN,CBB6RO25UB,NaN,Home Essentials,Bagasse,Kitchen Essentials,832,0,42723,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,119.0 INR,64.0,NaN,NaN,51.35,119.0,0.05,0.0,,,,,,,,,,,,,,,,,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
942,fuswn08080464,2026-04-29,2026-05-22,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin...",SPNH46S4G7HG3FWA,44191900.0,,ECO SOUL_140 mm Biodegradable_Soup Spoon_Beige...,45.0 INR,214.0 INR,,NaN,NaT,NaN,NaN,NaN,BS140MM50,NaN,Home Essentials,Bagasse,Tableware,100,0,4500,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,99.0 INR,100.0,NaN,NaN,44.85,99.0,0.05,0.0,,,,,,,,,,,,,,,,,2026-05-19 00:00:00,Fulfilled,2026-05-19 00:00:00
943,fuswn08080464,2026-04-29,2026-05-22,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart Ind

In [68]:
final_df['PO Creation Date'] = pd.to_datetime(
    final_df['PO Creation Date'],
    errors='coerce'
)
final_df['year_month'] = final_df['PO Creation Date'].dt.to_period('M').astype(str)
final_df['PO Creation Date'] = final_df['PO Creation Date'].dt.date


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\3998501918.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['PO Creation Date'] = pd.to_datetime(
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\3998501918.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['year_month'] = final_df['PO Creation Date'].dt.to_period('M').astype(str)
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\3998501918.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy o

In [69]:
final_df

,PO Number,PO Creation Date,PO Expiry Date,PO Delivered By Address,PO Delivered To,PO Item Code,HSN Code,PO Product UPC,PO Product Description,PO Basic Cost Price,PO Tax Amt,PO Landing Rate,Invoice Number,Invoice Date,invoice_dispatch_from,invoice_ship_to,invoice_bill_to,SKU,Invoice Item & Description,Category,Material,Sub Category,PO_quantity,Invoice_Quantity,PO_Amount,Invoice_Amount,Status,ClientID,Platform,Channel,SGST %,SGST Amt,CGST %,CGST Amt,IGST %,IGST Amt,PO MRP,Box/Case,invoice_amt_gst,Invoice Rate,NLC as per pricing sheet,MRP as per pricing sheet,GST as per pricing sheet,revenue,grn_item_code,grn_product_description,grn_mrp,grn_tax_amount,grn_quantity,grn_total_amount,grn_gmv_loss,dn_date,facility_name,dn_id,item_name,vendor_invoice_id,dn_quantity,dn_value,dn_reason,dn_remark,appt date,PO_fulfilled_status_from_file,requested_appt_date,year_month
10,fbhwn05718319,2025-01-09,2025-01-28,"Tower-B, 2nd Floor, Unit-B 202A, Sector 142, ,...","Flipkart India Private Limited, Sy no 18/2, 18...",CNSH5F7WSMGXZFQX,48236900.0,,ECO SOUL_Cup Set_White,42.0 INR,282.04 INR,,NaN,NaT,NaN,NaN,NaN,PCL5OZ50NL,NaN,Home Essentials,Paper,Kitchen Essentials,44,0,1848,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,99.0 INR,40.0,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,,,,,,,,,NaN,NaN,NaN,2025-01
11,fbhwn05769303,2025-01-21,2025-11-02,"Tower-B, 2nd Floor, Unit-B 202A, Sector 142, ,...","Flipkart India Private Limited, Sy no 18/2, 18...",CNSH5F7WSMGXZFQX,48236900.0,,ECO SOUL_Cup Set_White,42.0 INR,0.0 INR,,NaN,NaT,NaN,NaN,NaN,PCL5OZ50NL,NaN,Home Essentials,Paper,Kitchen Essentials,0,0,0,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,99.0 INR,40.0,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,,,,,,,,,NaN,NaN,NaN,2025-01
12,fbhwn05849736,2025-02-11,2025-02-28,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Sy no 18/2, 18...",GLSH27KKZE99D6KH,48236900.0,,ECO SOUL_Glass_White_100,116.0 INR,3964.7999999999997 INR,,NaN,NaT,NaN,NaN,NaN,PCL5OZ100NL,NaN,Home Essentials,Paper,Tableware,224,0,25984,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,599.0 INR,20.0,NaN,NaN,116.00,199.0,0.18,0.0,,,,,,,,,,,,,,,,,NaN,NaN,NaN,2025-02
13,fbhwn05849736,2025-02-11,2025-02-28,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Sy no 18/2, 18...",GLSH27KQJNDBMGYD,48236900.0,,ECO SOUL_Glass_White_50,116.0 INR,14160.0 INR,,NaN,NaT,NaN,NaN,NaN,PCL8OZ50NL,NaN,Home Essentials,Paper,Tableware,800,0,92800,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,549.0 INR,20.0,NaN,NaN,116.00,199.0,0.18,0.0,,,,,,,,,,,,,,,,,NaN,NaN,NaN,2025-02
14,fbhwn05849736,2025-02-11,2025-02-28,"1st Phase, Plot No. 162-A, Road Number 33 of K...","Flipkart India Private Limited, Sy no 18/2, 18...",BWLH5FZYW9XZEX9F,48237010.0,,ECO SOUL_180 ml_Serving Bowl_Beige_810103152366,51.35 INR,4576.0 INR,,NaN,NaT,NaN,NaN,NaN,CBB6RO25UB,NaN,Home Essentials,Bagasse,Kitchen Essentials,832,0,42723,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,119.0 INR,64.0,NaN,NaN,51.35,119.0,0.05,0.0,,,,,,,,,,,,,,,,,NaN,NaN,NaN,2025-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
942,fuswn08080464,2026-04-29,2026-05-22,"Billing Address: 305, Udyog Kendra, Extension ...","Flipkart India Private Limited, ESR Warehousin...",SPNH46S4G7HG3FWA,44191900.0,,ECO SOUL_140 mm Biodegradable_Soup Spoon_Beige...,45.0 INR,214.0 INR,,NaN,NaT,NaN,NaN,NaN,BS140MM50,NaN,Home Essentials,Bagasse,Tableware,100,0,4500,0,Unfulfilled,TH-1755734939046,QuickCommerce,Flikart,NaN,NaN,NaN,NaN,NaN,NaN,99.0 INR,100.0,NaN,NaN,44.85,99.0,0.05,0.0,,,,,,,,,,,,,,,,,2026-05-19 00:00:00,Fulfilled,2026-05-19 00:00:00,2026-04
943,fuswn08080464,2026-04-29,2026-05-22,"B

In [70]:
float_cols = final_df.select_dtypes(include='float')

final_df[float_cols.columns] = final_df[float_cols.columns].round(2)


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\3474924535.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df[float_cols.columns] = final_df[float_cols.columns].round(2)


In [71]:
import pandas as pd

final_df['year_month'] = pd.to_datetime(
    final_df['year_month'],
    errors='coerce'
)

final_df['year'] = final_df['year_month'].dt.year
final_df['month'] = final_df['year_month'].dt.strftime('%B')


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\3797021220.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['year_month'] = pd.to_datetime(
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\3797021220.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['year'] = final_df['year_month'].dt.year
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\3797021220.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

In [72]:
final_df['year'] = final_df['year'].astype(str)
final_df['month'] = final_df['month'].astype(str).str.zfill(2)

final_df['year_month'] = (
    final_df['year'] + '_' + final_df['month']
)
final_df['Invoice Date'] = final_df['Invoice Date'].dt.date


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\2757904007.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['year'] = final_df['year'].astype(str)
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\2757904007.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['month'] = final_df['month'].astype(str).str.zfill(2)
C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\2757904007.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Dat

In [73]:
final_df['Invoice Date'] = pd.to_datetime(final_df['Invoice Date']).dt.date


C:\Users\Shubham Verma\AppData\Local\Temp\ipykernel_35988\4016713438.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['Invoice Date'] = pd.to_datetime(final_df['Invoice Date']).dt.date


# Upload the file to SQL

In [74]:
import pandas as pd
import urllib.parse
import re
from sqlalchemy import create_engine, text, inspect

# ======================================================
# MYSQL CONFIG
# ======================================================
MYSQL_CONFIG = {
    "host": "192.168.50.148",
    "port": 3306,
    "user": "datahive_esh_test",
    "password": "Datahive@321!",
    "database": "datahive_esh_test",
}


DEPARTMENT = "quick_comm"
BASE_TABLE_NAME = "flipkart_orders"
TABLE_NAME = f"{DEPARTMENT}_{BASE_TABLE_NAME}"

# ======================================================
# CLEAN MYSQL COLUMN NAMES
# ======================================================
def clean_mysql_column(col):
    col = col.strip()
    col = col.replace(" ", "_")
    col = col.replace("%", "pct")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = col.replace("\\", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("[", "")
    col = col.replace("]", "")
    col = re.sub(r"__+", "_", col)
    col = re.sub(r"[^A-Za-z0-9_]", "", col)
    return col.lower()

# ======================================================
# CREATE MYSQL ENGINE
# ======================================================
password_encoded = urllib.parse.quote_plus(MYSQL_CONFIG["password"])

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_CONFIG['user']}:{password_encoded}"
    f"@{MYSQL_CONFIG['host']}:{MYSQL_CONFIG['port']}/{MYSQL_CONFIG['database']}",
    pool_pre_ping=True
)

# ======================================================
# TEST MYSQL CONNECTION (OPTIONAL BUT RECOMMENDED)
# ======================================================
def test_mysql_connection():
    print("\n🔍 Testing MySQL connection...")
    try:
        with engine.connect() as conn:
            db = conn.execute(text("SELECT DATABASE();")).fetchone()
            print("✅ Connected to database:", db[0])
    except Exception as e:
        print("❌ Connection failed:", e)
        return False
    return True

# ======================================================
# SAVE final_df TO MYSQL
# ======================================================
def save_final_df_to_mysql(final_df: pd.DataFrame):
    print("\n🟦 Uploading final_df to MySQL...")

    df = final_df.copy()
    print("📊 Rows to upload:", len(df))

    # Clean column names
    df.columns = [clean_mysql_column(c) for c in df.columns]

    inspector = inspect(engine)
    table_exists = inspector.has_table(TABLE_NAME)

    try:
        with engine.begin() as conn:

            if table_exists:
                print("🗑 Truncating existing table...")
                conn.execute(text(f"TRUNCATE TABLE `{TABLE_NAME}`"))
                mode = "append"
            else:
                print("📌 Creating new table...")
                mode = "fail"

            print("⬆ Inserting rows...")
            df.to_sql(
                TABLE_NAME,
                con=conn,
                if_exists=mode,
                index=False,
                chunksize=1000,
                method="multi"
            )

        print("✅ Upload completed successfully!")

    except Exception as e:
        print("❌ Upload failed:", e)

# ======================================================
# RUN
# ======================================================
# final_df MUST already exist in memory

if test_mysql_connection():
    save_final_df_to_mysql(final_df)



🔍 Testing MySQL connection...
✅ Connected to database: datahive_esh_test

🟦 Uploading final_df to MySQL...
📊 Rows to upload: 873
🗑 Truncating existing table...
⬆ Inserting rows...
✅ Upload completed successfully!
